<a href="https://colab.research.google.com/github/ambreenraheem/PGD_generative_AI_NED/blob/main/NGO_assistant_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

it is not working because it has no data of NGO's

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import chainlit as cl
from agents import (
    Agent,
    InputGuardrail,
    GuardrailFunctionOutput,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)
from agents.exceptions import InputGuardrailTripwireTriggered

# -----------------------------
# Setup
# -----------------------------

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Disable any tracing/telemetry by default (same as your engineering example)
set_tracing_disabled(disabled=True)

# Configure Gemini via OpenAI-compatible endpoint
external_client: AsyncOpenAI = AsyncOpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
llm_model: OpenAIChatCompletionsModel = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=external_client
)

# -----------------------------
# Guardrail Output Schema
# -----------------------------

class NGOInputCheck(BaseModel):
    """Guardrail classification for NGO assistant safety & scope checks."""
    topic: str = Field(..., description="One of: education, health, relief, livelihood, protection, volunteering, donations, complaints, general")
    collect_personal_data: bool = Field(..., description="User is trying to share or requesting to store personal identifiers (CNIC/NIC, phone, exact address) without consent form.")
    medical_diagnosis: bool = Field(..., description="User asks for medical diagnosis or prescription beyond general health info.")
    legal_advice: bool = Field(..., description="User asks for legal advice beyond general information.")
    unsafe_request: bool = Field(..., description="Violence, self-harm, hate, or otherwise unsafe content.")
    reasoning: str = Field(..., description="Why these flags were set. Keep concise.")

# -----------------------------
# Agents
# -----------------------------

# Guardrail classifier agent
guardrail_agent = Agent(
    name="NGO Guardrail Classifier",
    instructions=(
        "You are the safety and scope classifier for an NGO assistant. "
        "Given a user query, identify the most relevant NGO topic and whether the message "
        "contains restricted intents.\n\n"
        "TOPICS: education, health, relief, livelihood, protection, volunteering, donations, complaints, general.\n"
        "Flags:\n"
        "- collect_personal_data: true if user wants to give or asks you to store personal identifiers (CNIC/NIC, phone numbers, exact addresses) without consent workflow.\n"
        "- medical_diagnosis: true if user requests diagnosis or prescription (beyond general info and 'see a doctor' guidance).\n"
        "- legal_advice: true if user requests binding legal advice.\n"
        "- unsafe_request: true for violent, hateful, self-harm, or otherwise disallowed content.\n"
        "Respond with the NGOInputCheck schema only."
    ),
    output_type=NGOInputCheck,
    model=llm_model
)

# Program agents
education_agent = Agent(
    name="Education Program Agent",
    handoff_description="Scholarships, school enrollment, after-school tutoring, learning resources.",
    instructions=(
        "You are an NGO Education Program officer. Answer only about education: scholarships, "
        "enrollment, literacy classes, teacher training, school supplies. "
        "Give clear steps, documents required, locations, and contact timing if available. "
        "If the query is not education, defer."
    ),
    model=llm_model
)

health_agent = Agent(
    name="Health Program Agent",
    handoff_description="Medical camps, vaccination drives, health awareness sessions.",
    instructions=(
        "You are an NGO Health Program officer. Share general health camp information, vaccination schedules, "
        "screening camps, and referral pathways. Do NOT provide diagnosis or prescribe medications. "
        "Always suggest consulting a qualified clinician for medical concerns."
    ),
    model=llm_model
)

relief_agent = Agent(
    name="Relief Program Agent",
    handoff_description="Disaster relief, cash/food/NFI distribution, shelter support.",
    instructions=(
        "You are an NGO Disaster Relief officer. Explain eligibility, registration process, distribution points, "
        "helpline numbers, and verification needed for disaster assistance (floods, earthquakes, etc)."
    ),
    model=llm_model
)

livelihood_agent = Agent(
    name="Livelihoods Program Agent",
    handoff_description="Skills training, micro-grants, job placement support.",
    instructions=(
        "You are an NGO Livelihoods officer. Provide details on vocational training, micro-grants, "
        "entrepreneurship support, and job placement guidance. Include application steps and timelines."
    ),
    model=llm_model
)

protection_agent = Agent(
    name="Protection & Safeguarding Agent",
    handoff_description="GBV/child protection referrals, case management intake info.",
    instructions=(
        "You are an NGO Protection focal person. Provide trauma-informed, survivor-centered information. "
        "Share safe referral pathways and helplines. Avoid collecting sensitive details in chat."
    ),
    model=llm_model
)

volunteer_agent = Agent(
    name="Volunteer Management Agent",
    handoff_description="Volunteer sign-up, onboarding, training schedules.",
    instructions=(
        "You manage volunteers. Explain how to register, screening steps, orientation, and training schedules. "
        "Do not store personal identifiers in chat; direct users to official sign-up forms if needed."
    ),
    model=llm_model
)

donor_agent = Agent(
    name="Donor Support Agent",
    handoff_description="Donation methods, receipts, transparency, project earmarking.",
    instructions=(
        "You assist donors. Provide official donation channels, how to obtain receipts, and transparency reports. "
        "Never accept or request payment details in chat; direct them to secure portals only."
    ),
    model=llm_model
)

complaints_agent = Agent(
    name="Complaints & Feedback Agent",
    handoff_description="Grievance redressal mechanism (GRM), accountability, hotlines.",
    instructions=(
        "You handle complaints and feedback. Explain the GRM process, confidentiality, and escalation steps. "
        "Do not collect sensitive identifiers in chat; guide to the official complaint form or hotline."
    ),
    model=llm_model
)

# -----------------------------
# Guardrail function (InputGuardrail)
# -----------------------------

async def ngo_input_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    flags = result.final_output_as(NGOInputCheck)

    trip = False
    reasons = []

    if flags.unsafe_request:
        trip = True
        reasons.append("Unsafe content (violence/self-harm/hate).")
    if flags.collect_personal_data:
        trip = True
        reasons.append("Attempt to share/store personal identifiers without consent workflow.")
    if flags.medical_diagnosis:
        trip = True
        reasons.append("Medical diagnosis/prescription request.")
    if flags.legal_advice:
        trip = True
        reasons.append("Request for binding legal advice.")

    # Keep topic in context metadata for triage
    ctx.context["ngo_topic"] = flags.topic

    return GuardrailFunctionOutput(
        output_info=flags,
        tripwire_triggered=trip,
        tripwire_message="; ".join(reasons) if reasons else ""
    )

# -----------------------------
# Triage Agent (decides category, doesn't auto-handoff)
# -----------------------------

triage_agent = Agent(
    name="NGO Triage Agent",
    instructions=(
        "Decide the single best-matching category for the user's request. "
        "Respond with exactly one label from: education, health, relief, livelihood, protection, volunteering, donations, complaints, general."
    ),
    input_guardrails=[InputGuardrail(guardrail_function=ngo_input_guardrail)],
    model=llm_model
)

# -----------------------------
# Chainlit events & routing
# -----------------------------

CATEGORY_TO_AGENT = {
    "education": education_agent,
    "health": health_agent,
    "relief": relief_agent,
    "livelihood": livelihood_agent,
    "protection": protection_agent,
    "volunteering": volunteer_agent,
    "donations": donor_agent,
    "complaints": complaints_agent,
    # fallback for "general" -> education agent by default, could be a separate "Info Desk" agent
    "general": education_agent,
}

welcome_text = (
    "Welcome to the NGO Assistant!\n\n"
    "Ask about:\n"
    "• Education (scholarships, enrollment)\n"
    "• Health (medical/vaccination camps)\n"
    "• Relief (flood/earthquake assistance)\n"
    "• Livelihoods (skills training, micro-grants)\n"
    "• Protection (safe referrals)\n"
    "• Volunteering (join & trainings)\n"
    "• Donations (secure channels & receipts)\n"
    "• Complaints (GRM & hotlines)\n\n"
    "For your safety, do not share CNIC/phone/address here. We'll direct you to official forms when needed."
)

@cl.on_chat_start
async def on_chat_start():
    await cl.Message(content=welcome_text).send()

@cl.on_message
async def on_message(message: cl.Message):
    try:
        # Step 1: Triage with guardrails
        await cl.Message(content=f"**Triage** is analyzing your request: “{message.content}”").send()
        triage_result = await Runner.run(triage_agent, message.content)

        category = triage_result.final_output.strip().lower()
        topic_from_guardrail = triage_result.context.get("ngo_topic")
        # Prefer explicit category from triage; if missing, use guardrail topic
        category = category or topic_from_guardrail or "general"

        chosen_agent = CATEGORY_TO_AGENT.get(category, education_agent)

        # Step 2: Announce handoff
        await cl.Message(content=f"Handoff → **{chosen_agent.name}** (category: {category})").send()

        # Step 3: Get program response
        program_result = await Runner.run(chosen_agent, message.content)
        await cl.Message(content=f"**{chosen_agent.name}**:\n{program_result.final_output}").send()

    except InputGuardrailTripwireTriggered as e:
        # Provide safe, actionable redirection based on the flags
        reason = getattr(e, "args", ["Guardrail activated"])[0]
        safety_note = (
            "**Guardrail Activated**\n"
            f"Reason: {reason}\n\n"
            "You can still ask about our programs listed above. "
            "For personal data submission, medical concerns, or legal matters, "
            "we'll direct you to the correct official channels."
        )
        await cl.Message(content=safety_note).send()


Dockerfile

In [ ]:
# Use a specific and recent Python slim image for reproducibility and efficiency.
FROM python:3.10.13-slim

# The working directory is now set to the root of the container.
WORKDIR /

# Copy the requirements file first to optimize Docker's build cache.
COPY requirements.txt .

# Install dependencies from requirements.txt and also install openai-agents directly.
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt && \
    pip install --no-cache-dir openai-agents

# Copy the rest of your files directly into the container's root.
COPY . .

# Create and set permissions for necessary files/directories.
RUN mkdir -p .files .chainlit && \
    touch chainlit.md

# Create a non-root user for security.
RUN adduser --disabled-password --gecos "" chainlit
RUN chown -R chainlit:chainlit .files .chainlit chainlit.md
USER chainlit

# Set Chainlit environment variables.
ENV CHAINLIT_UI=True
ENV CHAINLIT_BROWSER_AUTO_OPEN=false

# Expose the port on which the application will run.
EXPOSE 7860

# Command to run the application.
CMD ["chainlit", "run", "app.py", "--host", "0.0.0.0", "--port", "7860"]

requirements.txt

In [ ]:
chainlit
python-dotenv
websockets
openai-agents

Variable Name in settings= CHAINLIT_RUN_MODULE\
app

Licence= Apache 2.0 license